# Notebook 32 — Final IMERG precipitation-convergence analysis

This is Phase 3. It makes no ERA5 or NASA network requests. It reads the compact event data backed up by Notebook 31, verifies that the collection plan is complete, then creates the four scatterplots and the correlation/regression table.

The predictor is the saved convergence for the digitized Shinoda JPCZ polygon. The two precipitation regions are the same Shinoda polygon and the additional coastal wedge.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
# Accept a raw Git URL even if it was accidentally pasted as a Markdown link.
if REPO_URL.startswith('[') and '](' in REPO_URL and REPO_URL.endswith(')'):
    REPO_URL = REPO_URL.rsplit('](', 1)[1][:-1]
if not REPO_URL.startswith('https://'):
    raise ValueError(f'REPO_URL must be a raw https Git URL, not {REPO_URL!r}')

from google.colab import drive
drive.mount('/content/drive')
if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    clone = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError(f'Git clone failed for {REPO_URL}:\n{clone.stderr}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from jpcz_catalog.imerg_workflow import association_statistics, atomic_csv, prepare_imerg_event_catalog, read_checkpoint

ALLOW_PARTIAL_ANALYSIS = False
DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
PLAN_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_collection_plan.csv'
EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
ANALYSIS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_precipitation_convergence_metrics.csv'
STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics.csv'
PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_scatter.png'

if not PLAN_PATH.exists():
    raise FileNotFoundError('The collection plan is missing. Run Notebook 30, then collect data with Notebook 31.')
plan = pd.read_csv(PLAN_PATH, parse_dates=['event_start', 'event_end', 'event_peak', 'precip_window_start', 'precip_window_end_exclusive'])
event_metrics = read_checkpoint(EVENT_PATH, parse_dates=('event_peak',))
if not {'event_id', 'event_peak'}.issubset(event_metrics.columns):
    event_metrics = pd.DataFrame(columns=['event_id', 'event_peak'])
analysis = plan.merge(event_metrics, on=['event_id', 'event_peak'], how='left')
atomic_csv(analysis, ANALYSIS_PATH)
complete_events = int((plan['collection_status'] == 'complete').sum())
all_data_ready = complete_events == len(plan) and len(plan) > 0
print(f'Drive event inventory: {complete_events}/{len(plan)} complete.')
print('Final-analysis readiness:', 'READY' if all_data_ready else 'NOT READY — resume Notebook 31.')
display(plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(analysis.head())

In [ ]:
specifications = []
for region, label in [('jpcz_polygon', 'Shinoda JPCZ polygon'), ('coastal_wedge', 'Coastal wedge')]:
    specifications.extend([
        (label, 'IMERG accumulation (mm)', 'jpcz_polygon_convergence_1e5_s-1', f'{region}_imerg_accumulation_mm'),
        (label, 'IMERG mean rate (mm h-1)', 'jpcz_polygon_convergence_1e5_s-1', f'{region}_imerg_mean_rate_mm_hr'),
    ])

if not all_data_ready and not ALLOW_PARTIAL_ANALYSIS:
    statistics_table = pd.DataFrame([
        {'region': region, 'precipitation_measure': measure, 'n': 0, 'status': 'waiting for complete Drive inventory'}
        for region, measure, _, _ in specifications
    ])
    print('Final statistics remain locked until all IMERG-eligible events are saved.')
else:
    statistics_table = pd.DataFrame([
        association_statistics(analysis, x_column=x, y_column=y, region=region, measure=measure)
        for region, measure, x, y in specifications
    ])
atomic_csv(statistics_table, STATS_PATH)
display(statistics_table.round(4))

In [ ]:
def plot_association(ax, x_column, y_column, title, ylabel, summary):
    if summary.get('status') != 'ok':
        ax.text(0.5, 0.5, 'Waiting for complete IMERG event inventory', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return None
    sample = analysis[[x_column, y_column, 'duration_hours']].dropna()
    points = ax.scatter(sample[x_column], sample[y_column], c=sample['duration_hours'], cmap='viridis', s=40, alpha=0.85, edgecolor='white', linewidth=0.35)
    fit = stats.linregress(sample[x_column], sample[y_column])
    xline = np.linspace(sample[x_column].min(), sample[x_column].max(), 100)
    ax.plot(xline, fit.intercept + fit.slope * xline, color='#c0392b', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Saved Shinoda-polygon convergence, -D12 (10^-5 s^-1)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.text(0.03, 0.97, f"n={int(summary['n'])}\nr={summary['pearson_r']:.2f} ({summary['r_95ci_low']:.2f}, {summary['r_95ci_high']:.2f})\np={summary['r_two_sided_p']:.3g}; {summary['null_decision_alpha_0.05']}", va='top', transform=ax.transAxes, fontsize=9, bbox={'facecolor': 'white', 'alpha': 0.9})
    return points

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
last_points = None
for ax, (region, measure, x_column, y_column) in zip(axes.flat, specifications):
    summary = statistics_table.loc[(statistics_table['region'] == region) & (statistics_table['precipitation_measure'] == measure)].iloc[0].to_dict()
    result = plot_association(ax, x_column, y_column, f'{region}: {measure}', measure, summary)
    if result is not None:
        last_points = result
if last_points is not None:
    fig.colorbar(last_points, ax=axes, shrink=0.82, pad=0.02, label='Merged-event duration (h)')
fig.suptitle('IMERG Final V07 precipitation versus saved JPCZ convergence', fontsize=15)
fig.savefig(PLOT_PATH, dpi=220, bbox_inches='tight')
plt.show()
print('Saved figure:', PLOT_PATH)

## Methods wording

For each merged JPCZ episode, we used the detector's saved 12-hour trailing, area-weighted 925-hPa divergence at the event peak and multiplied it by -1 so larger values indicate stronger convergence in the digitized Shinoda JPCZ polygon. We obtained GPM IMERG Final V07 gauge-calibrated precipitation (`precipitation`; half-hourly 0.1 degree grid, with a legacy `precipitationCal` fallback only if present), calculated cosine-latitude-area-weighted precipitation rates over the Shinoda polygon and coastal wedge, calculated event accumulation by summing rate times 0.5 hour, and calculated event mean rate by dividing by event-window duration. We evaluated associations with two-sided Pearson correlation and ordinary least-squares regression, reporting r, 95 percent confidence intervals, slope, R squared, and p values at alpha = 0.05.